In [1]:
import pandas as pd
import numpy as np

# --- 1. GENERATE LARGE, MESSY DATASET (120 Rows) ---

print("="*80)
print("--- STEP 1: Generating Large Messy E-commerce Order Data (N=120) ---")
print("="*80)

np.random.seed(42) # for reproducibility

N = 120
data = {
    'OrderID': list(range(1001, 1001 + N - 20)) + [1005, 1010] * 10, # Introduce 20 duplicates
    'CustomerID': [f'C-{i:03}' for i in np.random.randint(1, 80, N)],
    'OrderDate': pd.to_datetime(pd.Series(np.random.choice(pd.date_range('2024-04-01', '2024-06-30'), N-5))).astype(object).tolist() + ['2024/07/01', 'InvalidDate', '2024-06-01', np.nan, '2024-05-15'],
    'ItemPrice': np.random.uniform(10, 150, N).tolist(),
    'ShippingRegion': np.random.choice(['North', 'south', 'East', 'west', 'NORTH-EAST', 'N/A', 'NORTH'], N).tolist(),
    'ShippingCost': np.random.uniform(5, 20, N).tolist(),
    'Quantity': np.random.randint(1, 5, N).tolist(),
}

# Introduce specific errors for demonstration
data['ItemPrice'][5] = '99.5' # String price
data['ItemPrice'][15] = '$125.00' # Price with currency symbol
data['ItemPrice'][25] = 999.99 # Outlier
data['ItemPrice'][35] = 'N/A'  # String for missing price
# Convert to pandas Series to assign np.nan to a slice
shipping_cost_series = pd.Series(data['ShippingCost'])
shipping_cost_series[40:50] = np.nan # Introduce chunk of missing costs
data['ShippingCost'] = shipping_cost_series.tolist()


df_raw = pd.DataFrame(data)

# Print initial diagnostics
print("\n[DIAGNOSTICS BEFORE CLEANING]")
print(f"Total Rows: {len(df_raw)}")
print(f"Duplicate OrderIDs: {df_raw['OrderID'].duplicated().sum()}")
print(f"Missing Values:\n{df_raw.isnull().sum()}")
print(f"ItemPrice Dtypes (mix of numeric and string):\n{df_raw['ItemPrice'].apply(type).value_counts().head(2)}")
print("="*80)


# --- 2. THE CLEANING SOLUTION (RUNNING THE EXERCISE) ---

df_cleaned = df_raw.copy()

print("\n\n" + "="*80)
print("--- STEP 2: Executing Data Cleaning Tasks ---")
print("="*80)

# 1. UNIQUENESS (Task 2): Remove Duplicate Orders (OrderID is the unique identifier)
print("\n[UNIQUENESS] Task 2: Removing duplicates based on OrderID...")
df_cleaned.drop_duplicates(subset=['OrderID'], keep='first', inplace=True)
print(f"-> Rows after dropping duplicates: {len(df_cleaned)}")

# 2. VALIDITY (Task 4): Correct Data Types
print("\n[VALIDITY] Task 4: Converting 'ItemPrice' and 'OrderDate' to correct types...")

# A. Clean and convert 'ItemPrice'
df_cleaned['ItemPrice'] = df_cleaned['ItemPrice'].astype(str).str.replace('$', '', regex=False).replace(['N/A', 'Unknown'], np.nan)
df_cleaned['ItemPrice'] = pd.to_numeric(df_cleaned['ItemPrice'], errors='coerce')

# B. Convert 'OrderDate' (handles mixed formats and invalid strings)
df_cleaned['OrderDate'] = pd.to_datetime(df_cleaned['OrderDate'], errors='coerce', dayfirst=False)

# 3. COMPLETENESS (Task 1): Impute Missing ShippingCost
print("\n[COMPLETENESS] Task 1: Imputing missing 'ShippingCost' and 'ItemPrice'...")
# Use mean imputation for ShippingCost
mean_shipping = df_cleaned['ShippingCost'].mean()
df_cleaned['ShippingCost'].fillna(mean_shipping, inplace=True)
# For missing ItemPrice, we will fill with 0 since we can't infer price.
df_cleaned['ItemPrice'].fillna(0.00, inplace=True)

# 4. CONSISTENCY (Task 3): Standardize ShippingRegion
print("\n[CONSISTENCY] Task 3: Standardizing 'ShippingRegion' format...")
# Replace missing/invalid strings with NaN, convert to title case, and replace hyphens
df_cleaned['ShippingRegion'] = df_cleaned['ShippingRegion'].replace({'N/A': np.nan}).str.title().str.replace('-', ' ', regex=False)
# Impute remaining NaNs in ShippingRegion with 'Unknown'
df_cleaned['ShippingRegion'].fillna('Unknown', inplace=True)
print(f"-> Unique Regions after standardization: {df_cleaned['ShippingRegion'].unique()}")

# 5. ACCURACY (Task 5): Remove Outliers and Inconsistent Data
print("\n[ACCURACY] Task 5: Removing ItemPrice outliers (> $200) and null dates...")
initial_rows = len(df_cleaned)

# Remove prices > $200 (Outlier rule)
df_cleaned = df_cleaned[df_cleaned['ItemPrice'] <= 200]

# Remove rows where OrderDate failed conversion (Invalid data rule)
df_cleaned.dropna(subset=['OrderDate'], inplace=True)

rows_removed = initial_rows - len(df_cleaned)
print(f"-> Removed {rows_removed} rows due to price outliers or invalid dates.")


# --- 3. FINAL DIAGNOSTICS AND CLEANED DATA ---

print("\n\n" + "="*80)
print("--- STEP 3: Final Diagnostics on Cleaned Data ---")
print("="*80)

print(f"Total Final Rows: {len(df_cleaned)}")
print(f"Duplicate OrderIDs: {df_cleaned['OrderID'].duplicated().sum()}")
print(f"ItemPrice Dtype: {df_cleaned['ItemPrice'].dtype}")
print(f"OrderDate Dtype: {df_cleaned['OrderDate'].dtype}")
print(f"Missing Values:\n{df_cleaned.isnull().sum()}")
print("\nCleaned Data Head:")
print(df_cleaned[['OrderID', 'ItemPrice', 'ShippingRegion', 'OrderDate']].head(10).to_markdown(index=False))

--- STEP 1: Generating Large Messy E-commerce Order Data (N=120) ---

[DIAGNOSTICS BEFORE CLEANING]
Total Rows: 120
Duplicate OrderIDs: 20
Missing Values:
OrderID            0
CustomerID         0
OrderDate          1
ItemPrice          0
ShippingRegion     0
ShippingCost      10
Quantity           0
dtype: int64
ItemPrice Dtypes (mix of numeric and string):
ItemPrice
<class 'float'>    117
<class 'str'>        3
Name: count, dtype: int64


--- STEP 2: Executing Data Cleaning Tasks ---

[UNIQUENESS] Task 2: Removing duplicates based on OrderID...
-> Rows after dropping duplicates: 100

[VALIDITY] Task 4: Converting 'ItemPrice' and 'OrderDate' to correct types...

[COMPLETENESS] Task 1: Imputing missing 'ShippingCost' and 'ItemPrice'...

[CONSISTENCY] Task 3: Standardizing 'ShippingRegion' format...
-> Unique Regions after standardization: ['North East' 'South' 'West' 'Unknown' 'East' 'North']

[ACCURACY] Task 5: Removing ItemPrice outliers (> $200) and null dates...
-> Removed 1 rows d

/tmp/ipykernel_412/131864809.py:72: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned['ShippingCost'].fillna(mean_shipping, inplace=True)
/tmp/ipykernel_412/131864809.py:74: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=